# Section 2 — HUD ZIP3-to-county weighted mapping

**Purpose**: build a bridge from Fannie Mae's ZIP3 identifier to FEMA's county FIPS.

**Input**: `ZIP-COUNTY-FIPS_2017-06.csv` (5-digit ZIP → county FIPS, simplified crosswalk, no residential ratios).

**Output**: `zip3_county_weighted.parquet` — for each (ZIP3, county) pair, the weight = fraction of that ZIP3's 5-digit ZIPs that fall into the county.

**Note on the weighting**: the simplified crosswalk lacks residential-address ratios, so we weight by the count of 5-digit ZIPs. Chapter 3 of the dissertation acknowledges this as a measurement limitation.

This notebook is independent of Notebooks 01, 03, and 04.


## Setup

In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("your/data/path/here")
HUD_CROSSWALK = DATA_DIR / "ZIP-COUNTY-FIPS_2017-06.csv"
OUT_HUD_ZIP3  = DATA_DIR / "zip3_county_weighted.parquet"

print(f"Input:  {HUD_CROSSWALK}  (exists: {HUD_CROSSWALK.exists()})")
print(f"Output: {OUT_HUD_ZIP3}")

## Load HUD crosswalk

In [ ]:
crosswalk = pd.read_csv(
    HUD_CROSSWALK,
    dtype={'ZIP': str, 'STCOUNTYFP': str}
)

# Zero-pad IDs (STCOUNTYFP is 5 chars: 2-digit state + 3-digit county)
crosswalk['ZIP']        = crosswalk['ZIP'].str.zfill(5)
crosswalk['STCOUNTYFP'] = crosswalk['STCOUNTYFP'].str.zfill(5)

# Extract ZIP3 (first three digits of ZIP)
crosswalk['ZIP3'] = crosswalk['ZIP'].str[:3]

print(f"Total 5-digit ZIP rows: {len(crosswalk):,}")
print(f"Columns: {list(crosswalk.columns)}")
crosswalk.head()

## Build (ZIP3, county) weighted mapping

In [ ]:
# Count distinct 5-digit ZIPs per (ZIP3, county) pair
zip3_county = (
    crosswalk
    .groupby(['ZIP3', 'STCOUNTYFP', 'STATE'])
    .size()
    .reset_index(name='n_zips')
)

# Total 5-digit ZIPs per ZIP3 (across all counties it touches)
zip3_totals = (
    zip3_county
    .groupby('ZIP3')['n_zips']
    .sum()
    .rename('total_zips')
    .reset_index()
)

zip3_county = zip3_county.merge(zip3_totals, on='ZIP3')

# Weight = fraction of this ZIP3's 5-digit ZIPs that fall into this county
zip3_county['weight'] = zip3_county['n_zips'] / zip3_county['total_zips']

zip3_county.head(10)

## Sanity checks

In [ ]:
print(f"Total (ZIP3, county) pairs: {len(zip3_county):,}")
print(f"Unique ZIP3 areas: {zip3_county['ZIP3'].nunique():,}")
print(f"Average counties per ZIP3: {len(zip3_county) / zip3_county['ZIP3'].nunique():.2f}")
print()
print(f"Florida ZIP3 areas: {zip3_county[zip3_county['STATE']=='FL']['ZIP3'].nunique()}")
print(f"Florida (ZIP3, county) pairs: {(zip3_county['STATE']=='FL').sum()}")


In [ ]:
# Preview: which ZIP3 areas span the most counties in Florida?
fl = zip3_county[zip3_county['STATE'] == 'FL']
top_span = (
    fl.groupby('ZIP3').size()
    .sort_values(ascending=False)
    .head(10)
    .rename('counties_spanned')
)
print("Top Florida ZIP3 areas by county span:")
print(top_span.to_string())

## Save output

In [ ]:
zip3_county.to_parquet(OUT_HUD_ZIP3, index=False)
print(f"Saved: {OUT_HUD_ZIP3}")
print(f"Size:  {OUT_HUD_ZIP3.stat().st_size / 1e6:.2f} MB")